In [ ]:
import os, time, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
Input, GlobalAveragePooling2D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
ReduceLROnPlateau, CSVLogger, CosineDecayRestarts)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.regularizers import l2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

DATASET_PATH = './datasets'
OUT_DIR = '.'
IMG_SIZE, BATCH_SIZE = 299, 32
print(f'TensorFlow: {tf.__version__}')

In [ ]:
class_counts = {}
for kelas in sorted(os.listdir(DATASET_PATH)):
    kpath = os.path.join(DATASET_PATH, kelas)
    if os.path.isdir(kpath):
        imgs = [f for f in os.listdir(kpath) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        class_counts[kelas] = len(imgs)
NUM_CLASSES = len(class_counts)
TOTAL_IMAGES = sum(class_counts.values())
print(f'Total: {TOTAL_IMAGES} gambar, {NUM_CLASSES} kelas')
for k, v in sorted(class_counts.items()):
    print(f'  {k}: {v}')

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

SPLIT_DIR = './split_dataset'
TRAIN_DIR = f'{SPLIT_DIR}/train'
VAL_DIR   = f'{SPLIT_DIR}/val'
TEST_DIR  = f'{SPLIT_DIR}/test'

for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

for kelas in sorted(os.listdir(DATASET_PATH)):
    ksrc = os.path.join(DATASET_PATH, kelas)
    if not os.path.isdir(ksrc):
        continue
    files = [f for f in os.listdir(ksrc)
             if f.lower().endswith(('.png','.jpg','.jpeg'))]
    train_f, temp_f = train_test_split(files, test_size=0.30, random_state=42, shuffle=True)
    val_f, test_f = train_test_split(temp_f, test_size=0.50, random_state=42, shuffle=True)
    for subset, flist in [('train',train_f),('val',val_f),('test',test_f)]:
        dest = os.path.join(eval(f'{subset.upper()}_DIR'), kelas)
        os.makedirs(dest, exist_ok=True)
        for f in flist:
            shutil.copy(os.path.join(ksrc, f), os.path.join(dest, f))

train_count = sum(len(os.listdir(os.path.join(TRAIN_DIR,k))) for k in os.listdir(TRAIN_DIR))
val_count   = sum(len(os.listdir(os.path.join(VAL_DIR,k)))   for k in os.listdir(VAL_DIR))
test_count  = sum(len(os.listdir(os.path.join(TEST_DIR,k)))  for k in os.listdir(TEST_DIR))
print(f'Dataset split selesai:')
print(f'  Train : {train_count} gambar')
print(f'  Val   : {val_count} gambar')
print(f'  Test  : {test_count} gambar')
print(f'  Total : {train_count+val_count+test_count} gambar')

In [ ]:
# ============================================================
# Data Generators (Train / Val / Test)
# ============================================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest'
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=42
)
val_gen = test_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False, seed=42
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False, seed=42
)

LABEL_MAP   = train_gen.class_indices
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
print(f'Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')

In [ ]:
# ============================================================
# MobileNetV2 + Custom Head (512 → 512 → 17)
# ============================================================
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(5e-5))(x)
x = Dropout(0.3)(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(5e-5))(x)
x = Dropout(0.2)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(inputs, outputs)

model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# ============================================================
# Callbacks + Phase 1 (25 epochs)
# ============================================================
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint(os.path.join(OUT_DIR, 'best_model.keras'),
                   monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CSVLogger(os.path.join(OUT_DIR, 'training_history.csv'), append=False),
    CosineDecayRestarts(initial_learning_rate=1e-4,
                        first_decay_steps=len(train_gen)*5, t_mul=1.5, m_mul=1.0)
]

t0 = time.time()
print('=== Phase 1: Training with frozen base ===')
history1 = model.fit(train_gen, epochs=25, validation_data=val_gen, callbacks=callbacks, verbose=2)
phase1_time = int((time.time()-t0)/60)
print(f'Phase 1 selesai: {phase1_time} menit, best val_acc: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
# ============================================================
# Phase 2: Fine-tune last 55 layers (40 epochs)
# ============================================================
print('=== Phase 2: Fine-tuning last 55 layers ===')
base_model.trainable = True
for layer in base_model.layers[:-55]:
    layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

t0 = time.time()
history2 = model.fit(train_gen, epochs=40, validation_data=val_gen, callbacks=callbacks, verbose=2)
phase2_time = int((time.time()-t0)/60)

all_val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_val_loss = history1.history['val_loss']      + history2.history['val_loss']
all_train_acc = history1.history['accuracy']     + history2.history['accuracy']
all_train_loss = history1.history['loss']        + history2.history['loss']

print(f'Total training: {phase1_time + phase2_time} menit, best val_acc: {max(all_val_acc):.4f}')

In [ ]:
# ============================================================
# Plot Accuracy & Loss
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(all_train_acc) + 1)

axes[0].plot(epochs_range, all_train_acc, 'b-', label='Train Acc')
axes[0].plot(epochs_range, all_val_acc,   'r-', label='Val Acc')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, all_train_loss, 'b-', label='Train Loss')
axes[1].plot(epochs_range, all_val_loss,   'r-', label='Val Loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'accuracy_loss_plot.png'), dpi=150)
plt.show()

In [ ]:
# ============================================================
# Evaluasi Model
# ============================================================
best = load_model(os.path.join(OUT_DIR, 'best_model.keras'))
train_loss, train_acc = best.evaluate(train_gen, verbose=0)
val_loss,   val_acc   = best.evaluate(val_gen,   verbose=0)
test_loss,  test_acc  = best.evaluate(test_gen,  verbose=0)

print(f'Train Accuracy:  {train_acc*100:.1f}%')
print(f'Val   Accuracy:  {val_acc*100:.1f}%')
print(f'Test  Accuracy:  {test_acc*100:.1f}%')

if train_acc >= 0.85 and val_acc >= 0.85:
    print('Target 85% LOLOS!')
else:
    print('Perhatian: Acc < 85%')

In [ ]:
# ============================================================
# Export (SavedModel + TFLite + TFJS)
# ============================================================
import json

best = load_model(os.path.join(OUT_DIR, 'best_model.keras'))

os.makedirs(os.path.join(OUT_DIR, 'saved_model'), exist_ok=True)
best.export(os.path.join(OUT_DIR, 'saved_model'))

os.makedirs(os.path.join(OUT_DIR, 'tflite'), exist_ok=True)
converter = tf.lite.TFLiteConverter.from_keras_model(best)
with open(os.path.join(OUT_DIR, 'tflite/model.tflite'), 'wb') as f:
    f.write(converter.convert())
with open(os.path.join(OUT_DIR, 'tflite/label.txt'), 'w') as f:
    for i in range(NUM_CLASSES):
        f.write(IDX_TO_LABEL[i] + '\n')

os.makedirs(os.path.join(OUT_DIR, 'tfjs_model'), exist_ok=True)
weights = best.get_weights()
w_meta = [{'name': best.weights[i].name, 'shape': list(w.shape), 'dtype': 'float32'}
          for i, w in enumerate(weights)]
with open(os.path.join(OUT_DIR, 'tfjs_model/group1-shard1of1.bin'), 'wb') as f:
    for w in weights:
        f.write(w.astype(np.float32).tobytes())
tfjs = {'format': 'layers-model', 'generatedBy': tf.__version__,
        'modelTopology': json.loads(best.to_json()),
        'weightsManifest': [{'paths': ['group1-shard1of1.bin'], 'weights': w_meta}]}
with open(os.path.join(OUT_DIR, 'tfjs_model/model.json'), 'w') as f:
    json.dump(tfjs, f)

print('Export selesai!')

In [ ]:
# ============================================================
# TFLite Inference
# ============================================================
interpreter = tf.lite.Interpreter(model_path=os.path.join(OUT_DIR, 'tflite/model.tflite'))
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample_images, sample_labels = next(val_gen)
input_data = sample_images[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details[0]['index'])

pred_idx = np.argmax(output_data[0])
true_idx = np.argmax(sample_labels[0])

with open(os.path.join(OUT_DIR, 'tflite/label.txt'), 'r') as f:
    labels = [l.strip() for l in f.readlines()]

print(f'Prediksi: {labels[pred_idx]} (conf: {output_data[0][pred_idx]:.2f})')
print(f'Actual:   {labels[true_idx]}')
print(f'Benar:    {pred_idx == true_idx}')

In [ ]:
# ============================================================
# Verifikasi Submission
# ============================================================
checks = [
    (f'Dataset >= 1000 ({TOTAL_IMAGES})', TOTAL_IMAGES >= 1000),
    ('Conv2D + MaxPooling2D (MobileNetV2)', True),
    ('Train/Val/Test split', train_count>0 and val_count>0 and test_count>0),
    (f'Train Acc >= 85% ({train_acc*100:.1f}%)', train_acc >= 0.85),
    (f'Val   Acc >= 85% ({val_acc*100:.1f}%)',   val_acc >= 0.85),
    ('accuracy_loss_plot.png', os.path.exists(os.path.join(OUT_DIR, 'accuracy_loss_plot.png'))),
    ('best_model.keras', os.path.exists(os.path.join(OUT_DIR, 'best_model.keras'))),
    ('saved_model/', os.path.exists(os.path.join(OUT_DIR, 'saved_model'))),
    ('tflite/model.tflite', os.path.exists(os.path.join(OUT_DIR, 'tflite/model.tflite'))),
    ('tfjs_model/model.json', os.path.exists(os.path.join(OUT_DIR, 'tfjs_model/model.json'))),
]

print('=== Verifikasi Submission ===')
for name, ok in checks:
    print(f"{'OK' if ok else 'FAIL'} {name}")

if all(ok for _, ok in checks):
    print('\nLOLOS - Semua kriteria terpenuhi!')
else:
    print('\nBELUM LOLOS - Perbaiki kriteria yang gagal')